# Entrega P5: Modelo Multidimensional ROLAP y Diseño de ETL

**Maestría en Inteligencia Analítica de Datos (MIAD) • Universidad de los Andes**  
**Curso:** Modelado de Datos y ETL  
**Proyecto:** Caso RaSA (Ente Regulador de Aseguradoras de Salud de los Alpes)  
**Grupo:** 6  
**Base de Datos del Data Mart:** `Proyecto_202615_G6`  


**Integrantes del Equipo (Grupo 6)**

| Nombres y Apellidos | Correo UniAndes |
| :--- | :--- |
| Ximena Meléndez | c.melendezv@uniandes.edu.co |
| Camila Pérez Cárdenas | a.perez36@uniandes.edu.co |
| Daniel Felipe Pérez Jaimes | df.perezj@uniandes.edu.co |

## Conexión a la Base de Datos del Grupo _Proyecto_202615_G6_

### 1. Inicialización del Entorno, Conector JDBC y SparkSession

In [1]:
import os
import sys
import urllib.request
from pathlib import Path
try:
    import pymysql
except ImportError:
    %pip install -q pymysql
    import pymysql
try:
    import pyspark
except ImportError:
    %pip install -q pyspark==4.2.0 pandas==2.3.3 numpy==2.5.3 pyarrow==25.0.1
    import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 1. Detección de entorno
IN_CODESPACES = os.environ.get('CODESPACES') == 'true'
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ENVIRONMENT = 'colab' if IN_COLAB else 'codespaces' if IN_CODESPACES else 'local'
print(f"Entorno detectado: {ENVIRONMENT}")

# 2. Descarga y configuración del conector JDBC MySQL
CONNECTOR_VERSION = '9.7.0'
CONNECTOR_URL = (f'https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/'
                 f'{CONNECTOR_VERSION}/mysql-connector-j-{CONNECTOR_VERSION}.jar')

def obtener_jar_mysql():
    env_path = os.environ.get('CONNECTOR_J_PATH', '')
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    jar = Path.home() / '.miid4103' / 'jars' / f'mysql-connector-j-{CONNECTOR_VERSION}.jar'
    if not jar.exists():
        jar.parent.mkdir(parents=True, exist_ok=True)
        print(f"Descargando MySQL Connector/J {CONNECTOR_VERSION}...")
        parcial = jar.with_suffix('.part')
        urllib.request.urlretrieve(CONNECTOR_URL, parcial)
        parcial.replace(jar)
    return jar

MYSQL_JAR = obtener_jar_mysql()
MYSQL_JAR_URI = MYSQL_JAR.as_uri()

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ.setdefault('SPARK_HOME', os.path.dirname(pyspark.__file__))

# 3. Inicialización de SparkSession con el driver JDBC cargado
spark = (SparkSession.builder
         .master('local[*]')
         .appName('MIID4103-P5-ETL-Grupo6')
         .config('spark.driver.host', '127.0.0.1')
         .config('spark.jars', MYSQL_JAR_URI)
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.ui.enabled', 'false')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.ansi.enabled', 'false')
         .getOrCreate())

spark.sparkContext.setLogLevel('ERROR')
print(f"SparkSession lista con conector: {MYSQL_JAR.name}")

Entorno detectado: codespaces


Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
26/11/11 18:55:57 WARN Utils: Your hostname, codespaces-7ef8dd resolves to a loopback address: 127.0.0.1; using 10.0.3.152 instead (on interface eth0)
26/11/11 18:55:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/11/11 18:55:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


SparkSession lista con conector: mysql-connector-j.jar


### 2. Credenciales, Conexiones de BDs y Funciones de ETL

In [ ]:
# 1. Credenciales individuales de UniAndes
db_user = 'usuario'
db_psswd = 'contraseña'

# 2. Servidor MySQL UniAndes
db_host = '157.253.236.120'
db_port = 8080

# 3. Bases de Datos del Proyecto
db_origen = 'RaSaTransaccional'        # Fuente transaccional (Extract)
db_destino = 'Proyecto_202615_G6'      # Data Mart del Grupo 6 (Load)

# 4. URLs JDBC de conexión
jdbc_opciones = 'useConfigs=maxPerformance&connectTimeout=10000&socketTimeout=600000'
url_origen = f'jdbc:mysql://{db_host}:{db_port}/{db_origen}?{jdbc_opciones}'
url_destino = f'jdbc:mysql://{db_host}:{db_port}/{db_destino}?rewriteBatchedStatements=true&{jdbc_opciones}'

# 5. Función DDL segura con PyMySQL (CREATE / DROP / TRUNCATE TABLE)
def ejecutar_sql(query):
    """Ejecuta sentencias DDL directamente en la base de datos destino."""
    conn = pymysql.connect(
        host=db_host,
        port=db_port,
        user=db_user,
        password=db_psswd,
        database=db_destino,
        autocommit=True
    )
    try:
        with conn.cursor() as cursor:
            cursor.execute(query)
    finally:
        conn.close()

# 6. Función de Extracción (E): Lee tablas transaccionales desde RaSaTransaccional
def extraer_tabla(nombre_tabla):
    """Extrae una tabla desde RaSaTransaccional a un DataFrame de PySpark."""
    return (spark.read.format('jdbc')
            .option('url', url_origen)
            .option('dbtable', f'{db_origen}.{nombre_tabla}')
            .option('user', db_user)
            .option('password', db_psswd)
            .option('driver', 'com.mysql.cj.jdbc.Driver')
            .load())

# 7. Función de Carga (L): Escribe DataFrames transformados en Proyecto_202615_G6
def cargar_tabla(df, nombre_tabla, modo="append"):
    """Escribe un DataFrame de Spark en la base de datos destino con inserción en lotes."""
    (df.write.format('jdbc')
       .option('url', url_destino)
       .option('dbtable', f'{db_destino}.{nombre_tabla}')
       .option('user', db_user)
       .option('password', db_psswd)
       .option('driver', 'com.mysql.cj.jdbc.Driver')
       .option('batchsize', '5000')
       .mode(modo)
       .save())

print("Parámetros y funciones de Extracción, Carga y DDL configurados exitosamente.")

Parámetros y funciones de Extracción, Carga y DDL configurados exitosamente.


### 3. EXTRACCIÓN: Tablas fuente desde RaSaTransaccional

In [3]:
areas_servicio   = extraer_tabla('FuenteAreasDeServicio_Copia_E').cache()
condiciones_pago = extraer_tabla('FuenteCondicionesDePago_Copia_E').cache()
planes_beneficio = extraer_tabla('FuentePlanesBeneficio_Copia_E').cache()
tipos_beneficio  = extraer_tabla('FuenteTiposBeneficio_Copia_E').cache()

print("Extracción completada y en caché:")
print(f" - Áreas de Servicio:   {areas_servicio.count():,} registros")
print(f" - Condiciones de Pago: {condiciones_pago.count():,} registros")
print(f" - Planes de Beneficio: {planes_beneficio.count():,} registros")
print(f" - Tipos de Beneficio:  {tipos_beneficio.count():,} registros")

Extracción completada y en caché:
 - Áreas de Servicio:   188,815 registros
 - Condiciones de Pago: 31 registros
 - Planes de Beneficio: 36,036 registros
 - Tipos de Beneficio:  849 registros


## Fase 1: DDL en MySQL

### Creación de esquema estrella (DDL) en el Proyecto_202615_G6

Se eliminan las tabla de hechos y las otras 6 tablas (Por si se corre la celda otra vez para que no se dupliquen los datos) y luego se crean las 7 tablas con sus llaves primarias y restricciones _UNIQUE_ y las llaves antes de usar cargar_tabla

In [4]:
sentencias_ddl = [
    "DROP TABLE IF EXISTS Hecho_OfertaPlanBeneficio",
    "DROP TABLE IF EXISTS Dim_Tiempo",
    "DROP TABLE IF EXISTS Dim_Proveedor",
    "DROP TABLE IF EXISTS Dim_Plan",
    "DROP TABLE IF EXISTS Dim_TipoBeneficio",
    "DROP TABLE IF EXISTS Dim_AreaServicio",
    "DROP TABLE IF EXISTS Dim_CondicionPago",
    """
    CREATE TABLE Dim_Tiempo (
        FechaSK INT NOT NULL,
        Fecha DATE NOT NULL,
        Anio INT NOT NULL,
        Mes INT NOT NULL,
        NombreMes VARCHAR(20) NOT NULL,
        Dia INT NOT NULL,
        Semestre INT NOT NULL,
        Trimestre INT NOT NULL,
        PRIMARY KEY (FechaSK)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """,
    """
    CREATE TABLE Dim_Proveedor (
        ProveedorSK INT NOT NULL,
        IdProveedor_NK INT NOT NULL,
        NombreProveedor VARCHAR(150) NOT NULL,
        PRIMARY KEY (ProveedorSK),
        UNIQUE KEY uq_proveedor_nk (IdProveedor_NK)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """,
    """
    CREATE TABLE Dim_Plan (
        PlanSK INT NOT NULL,
        IdPlan_NK VARCHAR(50) NOT NULL,
        IdNivelServicio INT NOT NULL,
        DescripcionNivel VARCHAR(50) NOT NULL,
        PRIMARY KEY (PlanSK),
        UNIQUE KEY uq_plan_nivel (IdPlan_NK, IdNivelServicio)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """,
    """
    CREATE TABLE Dim_TipoBeneficio (
        TipoBeneficioSK INT NOT NULL,
        IdTipoBeneficio_NK VARCHAR(50) NOT NULL,
        NombreBeneficio VARCHAR(150) NOT NULL,
        UnidadLimite VARCHAR(50),
        EsBeneficioEsencial VARCHAR(10),
        SeguroLoCubre VARCHAR(10),
        TieneLimiteCuantitativo VARCHAR(10),
        ExcluidoMaxDentroRed VARCHAR(10),
        ExcluidoMaxFueraRed VARCHAR(10),
        FechaInicioValidez DATE NOT NULL,
        FechaFinValidez DATE NOT NULL,
        EsActual BOOLEAN NOT NULL,
        PRIMARY KEY (TipoBeneficioSK),
        UNIQUE KEY uq_beneficio_scd2 (IdTipoBeneficio_NK, FechaInicioValidez)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """,
    """
    CREATE TABLE Dim_AreaServicio (
        AreaServicioSK INT NOT NULL,
        IdAreaDeServicio_NK VARCHAR(50) NOT NULL,
        NombreAreaDeServicio VARCHAR(150),
        Estado VARCHAR(50),
        CondadoPrincipal VARCHAR(100),
        CondadosIncluidos TEXT,
        CantidadCondados INT,
        PoblacionActual BIGINT,
        AreaMillasCuadradas DOUBLE,
        DensidadPoblacional DOUBLE,
        PRIMARY KEY (AreaServicioSK),
        UNIQUE KEY uq_area_nk (IdAreaDeServicio_NK)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """,
    """
    CREATE TABLE Dim_CondicionPago (
        CondicionPagoSK INT NOT NULL,
        IdCondicionPago_NK INT NOT NULL,
        Tipo VARCHAR(50) NOT NULL,
        Descripcion VARCHAR(255),
        PRIMARY KEY (CondicionPagoSK),
        UNIQUE KEY uq_condicion_nk (IdCondicionPago_NK)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """,
    """
    CREATE TABLE Hecho_OfertaPlanBeneficio (
        FechaSK INT NOT NULL,
        ProveedorSK INT NOT NULL,
        PlanSK INT NOT NULL,
        TipoBeneficioSK INT NOT NULL,
        AreaServicioSK INT NOT NULL,
        CondicionCopagoSK INT NOT NULL,
        CondicionCoseguroSK INT NOT NULL,
        ValorCopago DOUBLE,
        ValorCoseguro DOUBLE,
        CantidadLimite INT,
        CantidadOfertas INT NOT NULL DEFAULT 1,
        INDEX idx_hecho_fecha (FechaSK),
        INDEX idx_hecho_proveedor (ProveedorSK),
        INDEX idx_hecho_plan (PlanSK),
        INDEX idx_hecho_beneficio (TipoBeneficioSK),
        INDEX idx_hecho_area (AreaServicioSK),
        INDEX idx_hecho_copago (CondicionCopagoSK),
        INDEX idx_hecho_coseguro (CondicionCoseguroSK),
        CONSTRAINT fk_hecho_tiempo FOREIGN KEY (FechaSK) REFERENCES Dim_Tiempo (FechaSK),
        CONSTRAINT fk_hecho_proveedor FOREIGN KEY (ProveedorSK) REFERENCES Dim_Proveedor (ProveedorSK),
        CONSTRAINT fk_hecho_plan FOREIGN KEY (PlanSK) REFERENCES Dim_Plan (PlanSK),
        CONSTRAINT fk_hecho_beneficio FOREIGN KEY (TipoBeneficioSK) REFERENCES Dim_TipoBeneficio (TipoBeneficioSK),
        CONSTRAINT fk_hecho_area FOREIGN KEY (AreaServicioSK) REFERENCES Dim_AreaServicio (AreaServicioSK),
        CONSTRAINT fk_hecho_copago FOREIGN KEY (CondicionCopagoSK) REFERENCES Dim_CondicionPago (CondicionPagoSK),
        CONSTRAINT fk_hecho_coseguro FOREIGN KEY (CondicionCoseguroSK) REFERENCES Dim_CondicionPago (CondicionPagoSK)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """
]

for idx, sql in enumerate(sentencias_ddl, start=1):
    ejecutar_sql(sql)

print(f"DDL ejecutado correctamente: {len(sentencias_ddl)} sentencias procesadas en {db_destino}.")

DDL ejecutado correctamente: 14 sentencias procesadas en Proyecto_202615_G6.


## Fase 2: Transformación y carga de las 6 Dimensiones

### Paso 1: Dim_Tiempo

In [5]:
import datetime

# T1 (Parte 1) Extraer valores distintos de Fecha desde planes_beneficio
fechas_crudas = planes_beneficio.select("Fecha").distinct()

# T2 Parsear formatos heterogéneos y estandarizar a fecha ISO (DATE)
fecha_limpia = F.regexp_replace(F.trim(F.col("Fecha")), r",\s+", ",")
fecha_iso = (
    F.when(fecha_limpia.rlike(r"^\d{4}-\d{2}-\d{2}$"), 
           F.to_date(fecha_limpia, "yyyy-MM-dd"))
     .when(fecha_limpia.rlike(r"^[A-Za-z]{3}\s+\d{1,2},\d{4}$"),
            F.to_date(fecha_limpia, "MMM dd,yyyy"))
     .otherwise(F.lit(None).cast("date"))
)

# T1 (Parte 2) Consolidar fechas ISO únicas y excluir 1920-12-31 y nulas del rango válido
fechas_validas = (
    fechas_crudas
    .select(fecha_iso.alias("Fecha"))
    .filter(F.col("Fecha").isNotNull() & (F.col("Fecha") != F.lit("1920-12-31").cast("date")))
    .distinct()
)

# T3 Calcular llave subrogada entera FechaSK en formato YYYYMMDD
dim_tiempo_t3 = fechas_validas.withColumn(
    "FechaSK", F.date_format(F.col("Fecha"), "yyyyMMdd").cast("int")
)

# T4  Derivar jerarquía temporal (Año, Mes, NombreMes, Dia)
meses_es = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

mapa_meses = F.create_map(*[F.lit(x) for k, v in meses_es.items() for x in (k, v)])

dim_tiempo_t4 = (
    dim_tiempo_t3
    .withColumn("Anio", F.year(F.col("Fecha")).cast("int"))
    .withColumn("Mes", F.month(F.col("Fecha")).cast("int"))
    .withColumn("NombreMes", mapa_meses[F.col("Mes")])
    .withColumn("Dia", F.dayofmonth(F.col("Fecha")).cast("int"))
)

# T5 Calcular periodos regulatorios (Semestre, Trimestre)
dim_tiempo_t5 = (
    dim_tiempo_t4
    .withColumn("Semestre", F.when(F.col("Mes") <= 6, F.lit(1)).otherwise(F.lit(2)).cast("int"))
    .withColumn("Trimestre", F.quarter(F.col("Fecha")).cast("int"))
    .select("FechaSK", "Fecha", "Anio", "Mes", "NombreMes", "Dia", "Semestre", "Trimestre")
)

# T6 Insertar registro comodín FechaSK = 0 ('1900-01-01', 'Desconocida') para 1920/nulas
schema_tiempo = StructType([
    StructField("FechaSK", IntegerType(), False),
    StructField("Fecha", DateType(), False),
    StructField("Anio", IntegerType(), False),
    StructField("Mes", IntegerType(), False),
    StructField("NombreMes", StringType(), False),
    StructField("Dia", IntegerType(), False),
    StructField("Semestre", IntegerType(), False),
    StructField("Trimestre", IntegerType(), False),
])

comodin_tiempo = spark.createDataFrame(
    [(0, datetime.date(1900, 1, 1), 1900, 1, "Desconocida", 1, 1, 1)],
    schema=schema_tiempo
)

dim_tiempo = comodin_tiempo.unionByName(dim_tiempo_t5).orderBy("FechaSK")
dim_tiempo.show(truncate=False)

# Limpia la tabla (mientras la de hechos esté vacía) para evitar duplicados al re-ejecutar
ejecutar_sql("DELETE FROM Dim_Tiempo")
cargar_tabla(dim_tiempo, "Dim_Tiempo")
print(f"Dim_Tiempo cargada: {dim_tiempo.count()} registros.")

+--------+----------+----+---+-----------+---+--------+---------+
|FechaSK |Fecha     |Anio|Mes|NombreMes  |Dia|Semestre|Trimestre|
+--------+----------+----+---+-----------+---+--------+---------+
|0       |1900-01-01|1900|1  |Desconocida|1  |1       |1        |
|20171231|2017-12-31|2017|12 |Diciembre  |31 |2       |4        |
|20181231|2018-12-31|2018|12 |Diciembre  |31 |2       |4        |
+--------+----------+----+---+-----------+---+--------+---------+

Dim_Tiempo cargada: 3 registros.


### Paso 2: Dim_Proveedor y Dim_Plan

#### 1. Dim_Proveedor

In [6]:
from pyspark.sql.window import Window

# T1. Extraer registros distintos de IdProveedor_T (IS NOT NULL)
proveedores_t1 = (
    planes_beneficio
    .select("IdProveedor_T")
    .filter(F.col("IdProveedor_T").isNotNull())
    .distinct()
)

# T2. Validar unicidad manteniendo tipo estricto INT (sin casteo a texto)
dim_proveedor_t2 = (
    proveedores_t1
    .select(F.col("IdProveedor_T").cast("int").alias("IdProveedor_NK"))
    .filter(F.col("IdProveedor_NK").isNotNull() & (F.col("IdProveedor_NK") != -1))
    .distinct()
)

# T3. Derivar etiqueta estándar: concat('Proveedor ', IdProveedor_T)
dim_proveedor_t3 = dim_proveedor_t2.withColumn(
    "NombreProveedor",
    F.concat(F.lit("Proveedor "), F.col("IdProveedor_NK").cast("string"))
)

# T4. Generar llave subrogada propia de bodega ProveedorSK (INT, PK)
dim_proveedor_t4 = (
    dim_proveedor_t3
    .withColumn(
        "ProveedorSK",
        F.row_number().over(Window.orderBy("IdProveedor_NK")).cast("int")
    )
    .select("ProveedorSK", "IdProveedor_NK", "NombreProveedor")
)

# T5. Insertar registro comodín ProveedorSK=0 (IdProveedor_NK=-1, 'Desconocido')
schema_proveedor = StructType([
    StructField("ProveedorSK", IntegerType(), False),
    StructField("IdProveedor_NK", IntegerType(), False),
    StructField("NombreProveedor", StringType(), False),
])

comodin_proveedor = spark.createDataFrame(
    [(0, -1, "Desconocido")],
    schema=schema_proveedor
)

dim_proveedor = comodin_proveedor.unionByName(dim_proveedor_t4).orderBy("ProveedorSK")
dim_proveedor.show(10, truncate=False)

# Limpia la tabla (mientras la de hechos esté vacía) para evitar duplicados al re-ejecutar
ejecutar_sql("DELETE FROM Dim_Proveedor")
cargar_tabla(dim_proveedor, "Dim_Proveedor")
print(f"Dim_Proveedor cargada: {dim_proveedor.count()} registros.")

+-----------+--------------+---------------+
|ProveedorSK|IdProveedor_NK|NombreProveedor|
+-----------+--------------+---------------+
|0          |-1            |Desconocido    |
|1          |10207         |Proveedor 10207|
|2          |11269         |Proveedor 11269|
|3          |11512         |Proveedor 11512|
|4          |12303         |Proveedor 12303|
|5          |12858         |Proveedor 12858|
|6          |14002         |Proveedor 14002|
|7          |14609         |Proveedor 14609|
|8          |15560         |Proveedor 15560|
|9          |15833         |Proveedor 15833|
+-----------+--------------+---------------+
only showing top 10 rows

Dim_Proveedor cargada: 126 registros.


#### 2. Dim_Plan

In [7]:
from pyspark.sql.window import Window

# T1. Filtrar registros con IdPlan_T IS NOT NULL y trim(IdPlan_T) <> ''
planes_t1 = planes_beneficio.filter(
    F.col("IdPlan_T").isNotNull() & (F.trim(F.col("IdPlan_T")) != "")
)

# T2. Extraer combinaciones únicas (IdPlan_T, IdNivelServicio_T) -> NK compuesta
dim_plan_t2 = (
    planes_t1
    .select(
        F.upper(F.trim(F.col("IdPlan_T"))).alias("IdPlan_NK"),
        F.col("IdNivelServicio_T").cast("int").alias("IdNivelServicio"),
    )
    .filter(
        (F.col("IdPlan_NK") != "DESCONOCIDO")
        & F.col("IdNivelServicio").isNotNull()
    )
    .distinct()
)

# T3. Asignar etiqueta neutra DescripcionNivel ('Nivel 1', 'Nivel 2', 'Nivel 3')
dim_plan_t3 = dim_plan_t2.withColumn(
    "DescripcionNivel",
    F.concat(F.lit("Nivel "), F.col("IdNivelServicio").cast("string"))
)

# T4. Generar llave subrogada propia de bodega PlanSK (INT, PK)
dim_plan_t4 = (
    dim_plan_t3
    .withColumn(
        "PlanSK",
        F.row_number().over(Window.orderBy("IdPlan_NK", "IdNivelServicio")).cast("int")
    )
    .select("PlanSK", "IdPlan_NK", "IdNivelServicio", "DescripcionNivel")
)

# T5. Insertar comodín PlanSK=0 (IdPlan_NK='DESCONOCIDO', IdNivelServicio=0)
schema_plan = StructType([
    StructField("PlanSK", IntegerType(), False),
    StructField("IdPlan_NK", StringType(), False),
    StructField("IdNivelServicio", IntegerType(), False),
    StructField("DescripcionNivel", StringType(), False),
])

comodin_plan = spark.createDataFrame(
    [(0, "DESCONOCIDO", 0, "Desconocido")],
    schema=schema_plan
)

dim_plan = comodin_plan.unionByName(dim_plan_t4).orderBy("PlanSK")
dim_plan.show(10, truncate=False)

# Limpia la tabla (mientras la de hechos esté vacía) para evitar duplicados al re-ejecutar
ejecutar_sql("DELETE FROM Dim_Plan")
cargar_tabla(dim_plan, "Dim_Plan")
print(f"Dim_Plan cargada: {dim_plan.count()} registros.")

+------+-----------------+---------------+----------------+
|PlanSK|IdPlan_NK        |IdNivelServicio|DescripcionNivel|
+------+-----------------+---------------+----------------+
|0     |DESCONOCIDO      |0              |Desconocido     |
|1     |10207VA0380001-06|1              |Nivel 1         |
|2     |10207VA0380001-06|2              |Nivel 2         |
|3     |10207VA0380001-06|3              |Nivel 3         |
|4     |10207VA0430006-01|1              |Nivel 1         |
|5     |10207VA0430006-01|2              |Nivel 2         |
|6     |10207VA0430006-01|3              |Nivel 3         |
|7     |10207VA0440004-00|1              |Nivel 1         |
|8     |10207VA0440004-00|2              |Nivel 2         |
|9     |10207VA0440004-00|3              |Nivel 3         |
+------+-----------------+---------------+----------------+
only showing top 10 rows

Dim_Plan cargada: 1168 registros.


### Paso 3: Dim_CondicionPago

In [8]:
from pyspark.sql.window import Window

# T1. Sanear Tipo='NaN' (ID 221 -> 'Copago') y 'SinTipoCopago' -> 'Copago' según desc.
tipo_limpio = F.trim(F.col("Tipo"))
condiciones_t1 = condiciones_pago.withColumn(
    "Tipo",
    F.when(
        tipo_limpio.isin("NaN", "SinTipoCopago") | F.col("Tipo").isNull(),
        F.when(F.lower(F.col("Descripcion")).contains("coinsurance"), F.lit("Coseguro"))
         .otherwise(F.lit("Copago")),
    ).otherwise(tipo_limpio),
)

# T2. Homologar 'Copagado' -> 'Copago' y 'Coseguridad' -> 'Coseguro' (dominio estricto)
mapa_tipos = F.create_map(
    F.lit("Copagado"), F.lit("Copago"),
    F.lit("Copago"), F.lit("Copago"),
    F.lit("Coseguridad"), F.lit("Coseguro"),
    F.lit("Coseguro"), F.lit("Coseguro"),
)
condiciones_t2 = condiciones_t1.withColumn(
    "Tipo",
    F.coalesce(mapa_tipos[F.col("Tipo")], F.lit("Copago")),
)

# T3. Deduplicar estrictamente por IdCondicionesDePago_T (NO por Tipo, Desc.)
condiciones_t3 = (
    condiciones_t2
    .select(
        F.col("IdCondicionesDePago_T").cast("int").alias("IdCondicionPago_NK"),
        F.col("Tipo"),
        F.col("Descripcion"),
    )
    .filter(F.col("IdCondicionPago_NK").isNotNull() & (F.col("IdCondicionPago_NK") != -1))
    .groupBy("IdCondicionPago_NK", "Tipo", "Descripcion")
    .count()
)

# T4. Resolver ID 204: conservar desc. modal ('Copay per Stay after deductible')
ventana_moda = Window.partitionBy("IdCondicionPago_NK").orderBy(
    F.col("count").desc(),
    F.length(F.col("Descripcion")).desc(),
    F.col("Descripcion").asc(),
)
condiciones_t4 = (
    condiciones_t3
    .withColumn("_rn", F.row_number().over(ventana_moda))
    .filter(F.col("_rn") == 1)
    .drop("count", "_rn")
)

# T5. Limpieza de espacios (trim) y estandarización textual en Descripcion
condiciones_t5 = (
    condiciones_t4
    .withColumn("Tipo", F.trim(F.col("Tipo")))
    .withColumn("Descripcion", F.regexp_replace(F.trim(F.col("Descripcion")), r"\s+", " "))
)

# T6. Generar CondicionPagoSK (INT, PK) con IdCondicionPago_NK (INT, UNIQUE)
condiciones_t6 = (
    condiciones_t5
    .withColumn(
        "CondicionPagoSK",
        F.row_number().over(Window.orderBy("IdCondicionPago_NK")).cast("int"),
    )
    .select("CondicionPagoSK", "IdCondicionPago_NK", "Tipo", "Descripcion")
)

# T7. Insertar registro comodín CondicionPagoSK=0 (IdCondicionPago_NK=-1)
schema_condicion = StructType([
    StructField("CondicionPagoSK", IntegerType(), False),
    StructField("IdCondicionPago_NK", IntegerType(), False),
    StructField("Tipo", StringType(), False),
    StructField("Descripcion", StringType(), True),
])

comodin_condicion = spark.createDataFrame(
    [(0, -1, "Desconocido", "Desconocido")],
    schema=schema_condicion,
)

dim_condicion_pago = comodin_condicion.unionByName(condiciones_t6).orderBy("CondicionPagoSK")
dim_condicion_pago.show(30, truncate=False)

# Limpia la tabla (mientras la de hechos esté vacía) para evitar duplicados al re-ejecutar
ejecutar_sql("DELETE FROM Dim_CondicionPago")
cargar_tabla(dim_condicion_pago, "Dim_CondicionPago")
print(f"Dim_CondicionPago cargada: {dim_condicion_pago.count()} registros.")

+---------------+------------------+-----------+--------------------------------+
|CondicionPagoSK|IdCondicionPago_NK|Tipo       |Descripcion                     |
+---------------+------------------+-----------+--------------------------------+
|0              |-1                |Desconocido|Desconocido                     |
|1              |9                 |Coseguro   |No Charge                       |
|2              |17                |Copago     |Copay per Day after deductible  |
|3              |18                |Coseguro   |No Charge after deductible      |
|4              |27                |Coseguro   |Coinsurance after deductible    |
|5              |34                |Copago     |Not Applicable                  |
|6              |36                |Coseguro   |Not Applicable                  |
|7              |45                |Coseguro   |Coinsurance                     |
|8              |51                |Copago     |No Charge                       |
|9              

### Paso 4: Dim_AreaServicio

In [9]:
from pyspark.sql.window import Window

# T1. Excluir solo IdAreaDeServicio_T IS NULL (6.288); NO filtrar Fecha=1800
area_t1 = (
    areas_servicio
    .filter(F.col("IdAreaDeServicio_T").isNotNull())
)

# T2. Corrección de signo en superficie: multiplicar Area negativa por -1
area_t2 = (
    area_t1
    .withColumn(
        "Area_limpia",
        F.when(F.col("Area") < 0, F.col("Area") * F.lit(-1.0))
        .otherwise(F.col("Area"))
        .cast("double")
    )
)

# T3. Truncar sufijo '0001' SOLO si PoblacionAct > 50M o >100x(Area*Densidad)
condicion_pob_anomala = (
    (F.col("PoblacionAct") > F.lit(50_000_000.0))
    | (
        F.col("Area_limpia").isNotNull()
        & F.col("Densidad").isNotNull()
        & (F.col("Area_limpia") > F.lit(0.0))
        & (F.col("Densidad") > F.lit(0.0))
        & (F.col("PoblacionAct") > (F.lit(100.0) * F.col("Area_limpia") * F.col("Densidad")))
    )
)

area_t3 = (
    area_t2
    .withColumn(
        "PoblacionAct_limpia",
        F.when(F.col("PoblacionAct").isNull(), F.lit(None).cast("bigint"))
        .when(
            condicion_pob_anomala,
            F.round(F.col("PoblacionAct").cast("double") / F.lit(10000.0), 0).cast("bigint")
        )
        .otherwise(F.round(F.col("PoblacionAct").cast("double"), 0).cast("bigint"))
    )
)

# T4. Deduplicar condados únicos por (IdAreaDeServicio_T, trim(Estado), trim(Condado))
area_t4_con_clave = (
    area_t3
    .withColumn("_estado_norm", F.coalesce(F.trim(F.col("Estado")), F.lit("")))
    .withColumn("_condado_norm", F.coalesce(F.trim(F.col("Condado")), F.lit("")))
)
w_dedup_geo = Window.partitionBy("IdAreaDeServicio_T", "_estado_norm", "_condado_norm").orderBy(
    F.when(F.col("Area_limpia").isNotNull() & (F.col("Area_limpia") > 0), 0).otherwise(1),
    F.when(F.col("PoblacionAct_limpia").isNotNull(), 0).otherwise(1),
    F.col("PoblacionAct_limpia").desc_nulls_last(),
    F.col("Area_limpia").desc_nulls_last(),
    F.col("NombreAreaDeServicio").asc_nulls_last()
)
area_t4 = (
    area_t4_con_clave
    .withColumn("_rn_geo", F.row_number().over(w_dedup_geo))
    .filter(F.col("_rn_geo") == 1)
    .drop("_rn_geo", "_estado_norm", "_condado_norm")
)

# T5. Consolidar 1 fila/IdAreaDeServicio_T: SUM(Pob, Area), Estado/Condado princ.
w_principal = Window.partitionBy("IdAreaDeServicio_T").orderBy(
    F.col("PoblacionAct_limpia").desc_nulls_last(),
    F.col("Area_limpia").desc_nulls_last(),
    F.col("Condado").asc_nulls_last(),
    F.col("NombreAreaDeServicio").asc_nulls_last()
)
area_t5 = (
    area_t4
    .withColumn("_rn_princ", F.row_number().over(w_principal))
    .groupBy("IdAreaDeServicio_T")
    .agg(
        F.coalesce(
            F.max(F.when(F.col("_rn_princ") == 1, F.substring(F.trim(F.col("NombreAreaDeServicio")), 1, 150))),
            F.first(F.substring(F.trim(F.col("NombreAreaDeServicio")), 1, 150), ignorenulls=True),
            F.lit("Área Sin Nombre")
        ).alias("NombreAreaDeServicio"),
        F.coalesce(
            F.max(F.when(F.col("_rn_princ") == 1, F.substring(F.trim(F.col("Estado")), 1, 50))),
            F.first(F.substring(F.trim(F.col("Estado")), 1, 50), ignorenulls=True),
            F.lit("Sin Estado")
        ).alias("Estado"),
        F.coalesce(
            F.max(F.when(F.col("_rn_princ") == 1, F.substring(F.trim(F.col("Condado")), 1, 100))),
            F.first(F.substring(F.trim(F.col("Condado")), 1, 100), ignorenulls=True),
            F.lit("Sin Condado")
        ).alias("CondadoPrincipal"),
        F.array_sort(F.collect_set(F.trim(F.col("Condado")))).alias("_condados_set"),
        F.count("*").cast("int").alias("_filas_geo"),
        F.sum("PoblacionAct_limpia").cast("bigint").alias("PoblacionActual"),
        F.round(F.sum("Area_limpia"), 4).cast("double").alias("AreaMillasCuadradas")
    )
    .withColumn("CantidadCondados", F.size(F.col("_condados_set")).cast("int"))
    .withColumn(
        "CondadosIncluidos",
        F.when(F.col("CantidadCondados") > 0, F.concat_ws(", ", F.col("_condados_set")))
        .otherwise(F.lit("Sin Condados"))
    )
)
dif_multi = area_t5.filter(F.col("CantidadCondados") != F.col("_filas_geo")).count()
print(f"Áreas con discrepancia entre CantidadCondados y _filas_geo: {dif_multi}")
area_t5 = area_t5.drop("_condados_set", "_filas_geo")

# T6. Calcular Densidad = Pob/Area SOLO cuando Area IS NOT NULL AND Area > 0
area_t6 = (
    area_t5
    .withColumn(
        "DensidadPoblacional",
        F.when(
            F.col("AreaMillasCuadradas").isNotNull()
            & (F.col("AreaMillasCuadradas") > 0)
            & F.col("PoblacionActual").isNotNull(),
            F.round(F.col("PoblacionActual").cast("double") / F.col("AreaMillasCuadradas"), 4)
        )
        .otherwise(F.lit(None).cast("double"))
        .cast("double")
    )
)

# T7. Generar AreaServicioSK (INT), cast NK VARCHAR(50) y comodín SK=0
w_sk_area = Window.orderBy(F.col("IdAreaDeServicio_T").asc())

area_t7_validos = (
    area_t6
    .withColumn("AreaServicioSK", F.row_number().over(w_sk_area).cast("int"))
    .withColumn("IdAreaDeServicio_NK", F.substring(F.col("IdAreaDeServicio_T").cast("string"), 1, 50))
    .select(
        "AreaServicioSK",
        "IdAreaDeServicio_NK",
        "NombreAreaDeServicio",
        "Estado",
        "CondadoPrincipal",
        "CondadosIncluidos",
        "CantidadCondados",
        "PoblacionActual",
        "AreaMillasCuadradas",
        "DensidadPoblacional"
    )
)

schema_area = StructType([
    StructField("AreaServicioSK", IntegerType(), False),
    StructField("IdAreaDeServicio_NK", StringType(), False),
    StructField("NombreAreaDeServicio", StringType(), True),
    StructField("Estado", StringType(), True),
    StructField("CondadoPrincipal", StringType(), True),
    StructField("CondadosIncluidos", StringType(), True),
    StructField("CantidadCondados", IntegerType(), True),
    StructField("PoblacionActual", LongType(), True),
    StructField("AreaMillasCuadradas", DoubleType(), True),
    StructField("DensidadPoblacional", DoubleType(), True),
])

comodin_area = spark.createDataFrame(
    [(0, "DESCONOCIDO", "Área No Catalogada", "Sin Estado", "Sin Condado", "Sin Condados", 0, 0, 0.0, 0.0)],
    schema=schema_area
)

dim_area_servicio = comodin_area.unionByName(area_t7_validos).cache()

ejecutar_sql("DELETE FROM Dim_AreaServicio")
cargar_tabla(dim_area_servicio, "Dim_AreaServicio")

dim_area_servicio.orderBy("AreaServicioSK").show(10, truncate=False)
print(f"Dim_AreaServicio cargada: {dim_area_servicio.count()} registros.")

Áreas con discrepancia entre CantidadCondados y _filas_geo: 0
+--------------+-------------------+-----------------------------+----------+---------------------+-------------------------------------------------------------------------------------------------------------------------+----------------+---------------+-------------------+-------------------+
|AreaServicioSK|IdAreaDeServicio_NK|NombreAreaDeServicio         |Estado    |CondadoPrincipal     |CondadosIncluidos                                                                                                        |CantidadCondados|PoblacionActual|AreaMillasCuadradas|DensidadPoblacional|
+--------------+-------------------+-----------------------------+----------+---------------------+-------------------------------------------------------------------------------------------------------------------------+----------------+---------------+-------------------+-------------------+
|0             |DESCONOCIDO        |Área No Catalogad

### Paso 5: Dim_TipoBeneficio

In [10]:
import datetime
from pyspark.sql.window import Window

# T1. Normalizar banderas ('Si'->'Yes', 'Nein'->'No') estrictamente a 'Yes'/'No'
def normalizar_bandera(col_name):
    val = F.upper(F.trim(F.col(col_name)))
    return (
        F.when(val.isin("YES", "SI", "SÍ", "S"), F.lit("Yes"))
        .when(val.isin("NO", "NEIN", "N"), F.lit("No"))
        .otherwise(F.lit("No"))
    )

benef_t1 = (
    tipos_beneficio
    .filter(F.col("IdTipoBeneficio_T").isNotNull() & F.col("Fecha").isNotNull())
    .withColumn("NombreBeneficio", F.substring(F.trim(F.col("Nombre")), 1, 150))
    .withColumn(
        "UnidadLimite",
        F.when(F.trim(F.col("UnidadDelLimite")) == "", F.lit(None).cast("string"))
        .otherwise(F.substring(F.trim(F.col("UnidadDelLimite")), 1, 50))
    )
    .withColumn("EsBeneficioEsencial", normalizar_bandera("EsEHB"))
    .withColumn("SeguroLoCubre", normalizar_bandera("EstaCubiertaPorSeguro"))
    .withColumn("TieneLimiteCuantitativo", normalizar_bandera("TieneLimiteCuantitativo"))
    .withColumn("ExcluidoMaxDentroRed", normalizar_bandera("ExcluidoDelDesembolsoMaximoDentroDeLaRed"))
    .withColumn("ExcluidoMaxFueraRed", normalizar_bandera("ExcluidoDelDesembolsoMaximoFueraDeLaRed"))
    .select(
        "IdTipoBeneficio_T",
        "Fecha",
        "NombreBeneficio",
        "UnidadLimite",
        "EsBeneficioEsencial",
        "SeguroLoCubre",
        "TieneLimiteCuantitativo",
        "ExcluidoMaxDentroRed",
        "ExcluidoMaxFueraRed"
    )
)

# T2. Consolidar exactamente 1 fila por (IdTipoBeneficio_T, Fecha) sin duplicados
benef_t2_sin_dup = benef_t1.dropDuplicates()

def moda_bandera_yes(col_name):
    votes_yes = F.sum(F.when(F.col(col_name) == "Yes", 1).otherwise(0))
    votes_no = F.sum(F.when(F.col(col_name) == "No", 1).otherwise(0))
    return F.when(votes_yes >= votes_no, F.lit("Yes")).otherwise(F.lit("No")).alias(col_name)

def moda_texto(df, group_cols, target_col):
    w_mode = Window.partitionBy(*group_cols).orderBy(
        F.col("_freq").desc(),
        F.col(target_col).asc()
    )
    return (
        df.filter(F.col(target_col).isNotNull())
        .groupBy(*group_cols, target_col)
        .agg(F.count("*").alias("_freq"))
        .withColumn("_rn", F.row_number().over(w_mode))
        .filter(F.col("_rn") == 1)
        .select(*group_cols, target_col)
    )

# T3. Desempate intra-anual (ej. ID 640): moda y, en empate, priorizar 'Yes'
benef_banderas_t3 = (
    benef_t2_sin_dup
    .groupBy("IdTipoBeneficio_T", "Fecha")
    .agg(
        moda_bandera_yes("EsBeneficioEsencial"),
        moda_bandera_yes("SeguroLoCubre"),
        moda_bandera_yes("TieneLimiteCuantitativo"),
        moda_bandera_yes("ExcluidoMaxDentroRed"),
        moda_bandera_yes("ExcluidoMaxFueraRed"),
    )
)

moda_nombre_t3 = moda_texto(benef_t2_sin_dup, ["IdTipoBeneficio_T", "Fecha"], "NombreBeneficio")
moda_unidad_t3 = moda_texto(benef_t2_sin_dup, ["IdTipoBeneficio_T", "Fecha"], "UnidadLimite")

benef_t3 = (
    benef_banderas_t3
    .join(moda_nombre_t3, on=["IdTipoBeneficio_T", "Fecha"], how="left")
    .join(moda_unidad_t3, on=["IdTipoBeneficio_T", "Fecha"], how="left")
    .withColumn("NombreBeneficio", F.coalesce(F.col("NombreBeneficio"), F.lit("Sin Nombre")))
)

# T4. SCD2 sin cambio 2017-2018: vigencia 2017-01-01 a 9999-12-31 (EsActual=True)
cols_scd2 = [
    "NombreBeneficio",
    "UnidadLimite",
    "EsBeneficioEsencial",
    "SeguroLoCubre",
    "TieneLimiteCuantitativo",
    "ExcluidoMaxDentroRed",
    "ExcluidoMaxFueraRed",
]

hash_scd2 = F.concat_ws(
    "||",
    *[F.coalesce(F.col(c), F.lit("__NULL__")) for c in cols_scd2]
)

w_scd2 = Window.partitionBy("IdTipoBeneficio_T").orderBy(F.col("Fecha").asc())

benef_t4 = (
    benef_t3
    .withColumn("_hash_scd2", hash_scd2)
    .withColumn("_prev_hash", F.lag("_hash_scd2").over(w_scd2))
    .filter(F.col("_prev_hash").isNull() | (F.col("_hash_scd2") != F.col("_prev_hash")))
    .drop("_hash_scd2", "_prev_hash")
)

# T5. SCD2 con cambio: 2017->..2017-12-31 (False); 2018->..9999-12-31 (True)
benef_t5 = (
    benef_t4
    .withColumn("_next_fecha", F.lead("Fecha").over(w_scd2))
    .withColumn(
        "FechaInicioValidez",
        F.to_date(F.concat(F.col("Fecha").cast("string"), F.lit("-01-01")), "yyyy-MM-dd")
    )
    .withColumn(
        "FechaFinValidez",
        F.when(
            F.col("_next_fecha").isNotNull(),
            F.to_date(F.concat((F.col("_next_fecha") - F.lit(1)).cast("string"), F.lit("-12-31")), "yyyy-MM-dd")
        )
        .otherwise(F.to_date(F.lit("9999-12-31"), "yyyy-MM-dd"))
    )
    .withColumn("EsActual", F.col("_next_fecha").isNull().cast("boolean"))
    .drop("_next_fecha")
)

# T6. Generar TipoBeneficioSK (INT, PK) y castear IdTipoBeneficio_NK VARCHAR(50)
w_sk_benef = Window.orderBy(F.col("IdTipoBeneficio_T").asc(), F.col("FechaInicioValidez").asc())

benef_t6 = (
    benef_t5
    .withColumn("TipoBeneficioSK", F.row_number().over(w_sk_benef).cast("int"))
    .withColumn("IdTipoBeneficio_NK", F.substring(F.col("IdTipoBeneficio_T").cast("string"), 1, 50))
    .select(
        "TipoBeneficioSK",
        "IdTipoBeneficio_NK",
        "NombreBeneficio",
        "UnidadLimite",
        "EsBeneficioEsencial",
        "SeguroLoCubre",
        "TieneLimiteCuantitativo",
        "ExcluidoMaxDentroRed",
        "ExcluidoMaxFueraRed",
        "FechaInicioValidez",
        "FechaFinValidez",
        "EsActual"
    )
)

# T7. Insertar registro comodín TipoBeneficioSK=0 (IdTipoBeneficio_NK='DESCONOCIDO')
schema_benef = StructType([
    StructField("TipoBeneficioSK", IntegerType(), False),
    StructField("IdTipoBeneficio_NK", StringType(), False),
    StructField("NombreBeneficio", StringType(), False),
    StructField("UnidadLimite", StringType(), True),
    StructField("EsBeneficioEsencial", StringType(), True),
    StructField("SeguroLoCubre", StringType(), True),
    StructField("TieneLimiteCuantitativo", StringType(), True),
    StructField("ExcluidoMaxDentroRed", StringType(), True),
    StructField("ExcluidoMaxFueraRed", StringType(), True),
    StructField("FechaInicioValidez", DateType(), False),
    StructField("FechaFinValidez", DateType(), False),
    StructField("EsActual", BooleanType(), False),
])

comodin_benef = spark.createDataFrame(
    [(
        0,
        "DESCONOCIDO",
        "Beneficio Desconocido",
        "Sin Unidad",
        "Sin Dato",
        "Sin Dato",
        "Sin Dato",
        "Sin Dato",
        "Sin Dato",
        datetime.date(1900, 1, 1),
        datetime.date(9999, 12, 31),
        False,
    )],
    schema=schema_benef
)

dim_tipo_beneficio = comodin_benef.unionByName(benef_t6).cache()

ejecutar_sql("DELETE FROM Dim_TipoBeneficio")
cargar_tabla(dim_tipo_beneficio, "Dim_TipoBeneficio")

dim_tipo_beneficio.orderBy("TipoBeneficioSK").show(15, truncate=False)
print(f"Dim_TipoBeneficio cargada: {dim_tipo_beneficio.count()} registros.")
print("Versiones por EsActual:")
dim_tipo_beneficio.groupBy("EsActual").count().show()
print("Verificación ID 640:")
dim_tipo_beneficio.filter(F.col("IdTipoBeneficio_NK") == "640").show(truncate=False)

+---------------+------------------+-------------------------------------------------------+---------------------------+-------------------+-------------+-----------------------+--------------------+-------------------+------------------+---------------+--------+
|TipoBeneficioSK|IdTipoBeneficio_NK|NombreBeneficio                                        |UnidadLimite               |EsBeneficioEsencial|SeguroLoCubre|TieneLimiteCuantitativo|ExcluidoMaxDentroRed|ExcluidoMaxFueraRed|FechaInicioValidez|FechaFinValidez|EsActual|
+---------------+------------------+-------------------------------------------------------+---------------------------+-------------------+-------------+-----------------------+--------------------+-------------------+------------------+---------------+--------+
|0              |DESCONOCIDO       |Beneficio Desconocido                                  |Sin Unidad                 |Sin Dato           |Sin Dato     |Sin Dato               |Sin Dato            |Sin Dato 

## Fase 3: Transformación y carga de la Tabla de Hechos (`Hecho_OfertaPlanBeneficio`)

### Paso 6: Hecho_OfertaPlanBeneficio

In [11]:
# Lookups limpios de las 6 dimensiones (excluyendo comodín SK=0 para controlar mapeo a 0)
lk_tiempo = dim_tiempo.filter(F.col("FechaSK") > 0).select(
    F.col("Fecha").alias("_dim_fecha"),
    F.col("FechaSK").alias("_fecha_sk")
)

lk_plan = dim_plan.filter(F.col("PlanSK") > 0).select(
    F.col("IdPlan_NK").alias("_dim_plan_nk"),
    F.col("IdNivelServicio").alias("_dim_nivel"),
    F.col("PlanSK").alias("_plan_sk")
)

lk_benef_rango = dim_tipo_beneficio.filter(F.col("TipoBeneficioSK") > 0).select(
    F.col("IdTipoBeneficio_NK").alias("_dim_ben_nk_r"),
    "FechaInicioValidez",
    "FechaFinValidez",
    F.col("TipoBeneficioSK").alias("_ben_sk_r"),
    F.col("TieneLimiteCuantitativo").alias("_lim_r")
)

lk_benef_actual = dim_tipo_beneficio.filter(
    (F.col("TipoBeneficioSK") > 0) & (F.col("EsActual") == True)
).select(
    F.col("IdTipoBeneficio_NK").alias("_dim_ben_nk_a"),
    F.col("TipoBeneficioSK").alias("_ben_sk_a"),
    F.col("TieneLimiteCuantitativo").alias("_lim_a")
)

lk_area = dim_area_servicio.filter(F.col("AreaServicioSK") > 0).select(
    F.col("IdAreaDeServicio_NK").alias("_dim_area_nk"),
    F.col("AreaServicioSK").alias("_area_sk")
)

lk_proveedor = dim_proveedor.filter(F.col("ProveedorSK") > 0).select(
    F.col("IdProveedor_NK").alias("_dim_prov_nk"),
    F.col("ProveedorSK").alias("_prov_sk")
)

lk_copago = dim_condicion_pago.filter(F.col("CondicionPagoSK") > 0).select(
    F.col("IdCondicionPago_NK").alias("_dim_cop_nk"),
    F.col("CondicionPagoSK").alias("_cop_sk")
)

lk_coseguro = dim_condicion_pago.filter(F.col("CondicionPagoSK") > 0).select(
    F.col("IdCondicionPago_NK").alias("_dim_cos_nk"),
    F.col("CondicionPagoSK").alias("_cos_sk")
)

# T1. Excluir duplicados (8.493), IdPlan vacío/'' y IdTipoBeneficio nulo/huérfano
hecho_t1 = (
    planes_beneficio
    .dropDuplicates()
    .filter(F.col("IdPlan_T").isNotNull() & (F.trim(F.col("IdPlan_T")) != ""))
    .filter(F.col("IdTipoBeneficio_T").isNotNull())
    .join(
        lk_benef_actual.select("_dim_ben_nk_a"),
        F.col("IdTipoBeneficio_T").cast("string") == F.col("_dim_ben_nk_a"),
        how="left_semi"
    )
)

# T2. Conservar IdAreaDeServicio_T nulo (2.041) y huérfano (1.098) para SK=0
hecho_t2 = (
    hecho_t1
    .withColumn("_nk_plan", F.upper(F.trim(F.col("IdPlan_T"))))
    .withColumn("_nk_nivel", F.col("IdNivelServicio_T").cast("int"))
    .withColumn("_nk_beneficio", F.col("IdTipoBeneficio_T").cast("string"))
    .withColumn("_nk_area", F.col("IdAreaDeServicio_T").cast("string"))
    .withColumn("_nk_proveedor", F.col("IdProveedor_T").cast("int"))
    .withColumn("_nk_copago", F.col("IdCondicionDePagoCopago_T").cast("int"))
    .withColumn("_nk_coseguro", F.col("IdCondicionDePagoCoseguro_T").cast("int"))
)

# T3. Estandarizar Fecha (1920-12-31->FechaSK=0) y cast DOUBLE copago/coseguro
fecha_limpia_h = F.regexp_replace(F.trim(F.col("Fecha")), r",\s+", ",")
fecha_iso_h = (
    F.when(fecha_limpia_h.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(fecha_limpia_h, "yyyy-MM-dd"))
     .when(fecha_limpia_h.rlike(r"^[A-Za-z]{3}\s+\d{1,2},\d{4}$"), F.to_date(fecha_limpia_h, "MMM dd,yyyy"))
     .otherwise(F.lit(None).cast("date"))
)

hecho_t3 = (
    hecho_t2
    .withColumn("_fecha_iso", fecha_iso_h)
    .join(lk_tiempo, F.col("_fecha_iso") == F.col("_dim_fecha"), how="left")
    .withColumn(
        "FechaSK",
        F.when(F.col("_fecha_iso") == F.lit("1920-12-31").cast("date"), F.lit(0))
         .otherwise(F.coalesce(F.col("_fecha_sk"), F.lit(0)))
         .cast("int")
    )
    .withColumn("ValorCopago", F.col("valorCopago").cast("double"))
    .withColumn("ValorCoseguro", F.col("valorCoseguro").cast("double"))
    .drop("_dim_fecha", "_fecha_sk")
)

# T4. JOIN Dim_Plan (IdPlan_T Y IdNivelServicio_T) y Dim_TipoBeneficio (SCD2)
hecho_t4 = (
    hecho_t3
    .join(
        lk_plan,
        (F.col("_nk_plan") == F.col("_dim_plan_nk"))
        & (F.col("_nk_nivel") == F.col("_dim_nivel")),
        how="left"
    )
    .join(
        lk_benef_rango,
        (F.col("_nk_beneficio") == F.col("_dim_ben_nk_r"))
        & (F.col("FechaSK") > 0)
        & (F.col("_fecha_iso") >= F.col("FechaInicioValidez"))
        & (F.col("_fecha_iso") <= F.col("FechaFinValidez")),
        how="left"
    )
    .join(
        lk_benef_actual,
        F.col("_nk_beneficio") == F.col("_dim_ben_nk_a"),
        how="left"
    )
    .withColumn("PlanSK", F.coalesce(F.col("_plan_sk"), F.lit(0)).cast("int"))
    .withColumn("TipoBeneficioSK", F.coalesce(F.col("_ben_sk_r"), F.col("_ben_sk_a"), F.lit(0)).cast("int"))
    .withColumn("_tiene_limite", F.coalesce(F.col("_lim_r"), F.col("_lim_a")))
    .drop(
        "_dim_plan_nk", "_dim_nivel", "_plan_sk",
        "_dim_ben_nk_r", "FechaInicioValidez", "FechaFinValidez", "_ben_sk_r", "_lim_r",
        "_dim_ben_nk_a", "_ben_sk_a", "_lim_a"
    )
)

# T5. Mapear restantes (área/condición/proveedor) no hallados o nulos a SK=0
hecho_t5 = (
    hecho_t4
    .join(lk_area, F.col("_nk_area") == F.col("_dim_area_nk"), how="left")
    .join(lk_proveedor, F.col("_nk_proveedor") == F.col("_dim_prov_nk"), how="left")
    .join(lk_copago, F.col("_nk_copago") == F.col("_dim_cop_nk"), how="left")
    .join(lk_coseguro, F.col("_nk_coseguro") == F.col("_dim_cos_nk"), how="left")
    .withColumn("AreaServicioSK", F.coalesce(F.col("_area_sk"), F.lit(0)).cast("int"))
    .withColumn("ProveedorSK", F.coalesce(F.col("_prov_sk"), F.lit(0)).cast("int"))
    .withColumn("CondicionCopagoSK", F.coalesce(F.col("_cop_sk"), F.lit(0)).cast("int"))
    .withColumn("CondicionCoseguroSK", F.coalesce(F.col("_cos_sk"), F.lit(0)).cast("int"))
    .drop(
        "_dim_area_nk", "_area_sk",
        "_dim_prov_nk", "_prov_sk",
        "_dim_cop_nk", "_cop_sk",
        "_dim_cos_nk", "_cos_sk"
    )
)

# T6. Imputar CantidadLimite=333 SOLO si TieneLimiteCuantitativo='Yes' y nulo/0
hecho_t6 = (
    hecho_t5
    .withColumn(
        "CantidadLimite",
        F.when(
            (F.col("_tiene_limite") == "Yes")
            & (F.col("cantidadLimite").isNull() | (F.col("cantidadLimite") <= 0)),
            F.lit(333)
        )
        .otherwise(F.col("cantidadLimite"))
        .cast("int")
    )
)

# T7. Validar topes (Copago<=3500, Coseguro<=100%) y fijar CantidadOfertas=1
hecho_oferta_plan_beneficio = (
    hecho_t6
    .filter(
        (F.col("ValorCopago").isNull() | ((F.col("ValorCopago") >= 0) & (F.col("ValorCopago") <= 3500)))
        & (F.col("ValorCoseguro").isNull() | ((F.col("ValorCoseguro") >= 0) & (F.col("ValorCoseguro") <= 100)))
    )
    .withColumn("CantidadOfertas", F.lit(1).cast("int"))
    .select(
        "FechaSK",
        "ProveedorSK",
        "PlanSK",
        "TipoBeneficioSK",
        "AreaServicioSK",
        "CondicionCopagoSK",
        "CondicionCoseguroSK",
        "ValorCopago",
        "ValorCoseguro",
        "CantidadLimite",
        "CantidadOfertas"
    )
    .cache()
)

ejecutar_sql("DELETE FROM Hecho_OfertaPlanBeneficio")
cargar_tabla(hecho_oferta_plan_beneficio, "Hecho_OfertaPlanBeneficio")

hecho_oferta_plan_beneficio.show(10, truncate=False)
print(f"Hecho_OfertaPlanBeneficio cargada: {hecho_oferta_plan_beneficio.count()} registros.")
print(f" - Con FechaSK = 0 (1920/nula):       {hecho_oferta_plan_beneficio.filter(F.col('FechaSK') == 0).count()}")
print(f" - Con AreaServicioSK = 0 (nula/huérf.): {hecho_oferta_plan_beneficio.filter(F.col('AreaServicioSK') == 0).count()}")
print(f" - Con CantidadLimite = 333 (imputada):  {hecho_oferta_plan_beneficio.filter(F.col('CantidadLimite') == 333).count()}")
print(f" - Con CantidadLimite NULL (ilimitados): {hecho_oferta_plan_beneficio.filter(F.col('CantidadLimite').isNull()).count()}")

+--------+-----------+------+---------------+--------------+-----------------+-------------------+-----------+-------------+--------------+---------------+
|FechaSK |ProveedorSK|PlanSK|TipoBeneficioSK|AreaServicioSK|CondicionCopagoSK|CondicionCoseguroSK|ValorCopago|ValorCoseguro|CantidadLimite|CantidadOfertas|
+--------+-----------+------+---------------+--------------+-----------------+-------------------+-----------+-------------+--------------+---------------+
|20171231|107        |985   |76             |4340          |5                |6                  |0.0        |0.0          |30            |1              |
|20171231|107        |982   |70             |4279          |21               |7                  |0.0        |0.0          |25            |1              |
|20181231|74         |709   |187            |0             |21               |7                  |0.0        |0.0          |90            |1              |
|20181231|63         |653   |177            |3146          |21  

## Fase 4: Verificación de Integridad en MySQL y Consultas Analíticas

En esta sección se audita directamente la bodega `Proyecto_202615_G6` en MySQL para comprobar la paridad de registros, el enlace histórico SCD Tipo 2 y la capacidad analítica del esquema estrella frente a los cuatro frentes de negocio de RaSA.

### 1. Auditoría de Conteos, Comodines y SCD2 en MySQL

In [12]:
def consultar_mysql(titulo, query):
    conn = pymysql.connect(
        host=db_host,
        port=db_port,
        user=db_user,
        password=db_psswd,
        database=db_destino,
    )
    try:
        with conn.cursor() as cursor:
            cursor.execute(query)
            cols = [desc[0] for desc in cursor.description]
            rows = cursor.fetchall()
    finally:
        conn.close()

    print(f"\n=== {titulo} ===")
    if not rows:
        print("(0 filas)")
        return
    anchos = [
        max([len(str(c))] + [len(str(r[i])) for r in rows])
        for i, c in enumerate(cols)
    ]
    encabezado = " | ".join(str(c).ljust(anchos[i]) for i, c in enumerate(cols))
    print(encabezado)
    print("-" * len(encabezado))
    for r in rows:
        print(" | ".join(str(r[i]).ljust(anchos[i]) for i in range(len(cols))))


consultar_mysql(
    "1. Auditoría de conteos en Proyecto_202615_G6",
    """
    SELECT 'Dim_Tiempo' AS Tabla, COUNT(*) AS Registros_MySQL, 3 AS Esperado FROM Dim_Tiempo
    UNION ALL
    SELECT 'Dim_Proveedor', COUNT(*), 126 FROM Dim_Proveedor
    UNION ALL
    SELECT 'Dim_Plan', COUNT(*), 1168 FROM Dim_Plan
    UNION ALL
    SELECT 'Dim_CondicionPago', COUNT(*), 24 FROM Dim_CondicionPago
    UNION ALL
    SELECT 'Dim_AreaServicio', COUNT(*), 5410 FROM Dim_AreaServicio
    UNION ALL
    SELECT 'Dim_TipoBeneficio', COUNT(*), 218 FROM Dim_TipoBeneficio
    UNION ALL
    SELECT 'Hecho_OfertaPlanBeneficio', COUNT(*), 22963 FROM Hecho_OfertaPlanBeneficio
    """
)

consultar_mysql(
    "2. Verificación de métricas y comodines en Hecho_OfertaPlanBeneficio",
    """
    SELECT
        COUNT(*) AS TotalFilas,
        SUM(CantidadOfertas) AS TotalOfertas,
        SUM(CASE WHEN FechaSK = 0 THEN 1 ELSE 0 END) AS Fecha_Comodin,
        SUM(CASE WHEN AreaServicioSK = 0 THEN 1 ELSE 0 END) AS Area_Comodin,
        SUM(CASE WHEN CantidadLimite = 333 THEN 1 ELSE 0 END) AS Limite_333_Imputado,
        SUM(CASE WHEN CantidadLimite IS NULL THEN 1 ELSE 0 END) AS Limite_NULL_Ilimitado,
        SUM(CASE WHEN CantidadLimite IS NOT NULL AND CantidadLimite <> 333 THEN 1 ELSE 0 END) AS Limite_Explicito
    FROM Hecho_OfertaPlanBeneficio
    """
)

consultar_mysql(
    "3. Verificación SCD Tipo 2 en MySQL (ID = 640)",
    """
    SELECT
        tb.TipoBeneficioSK,
        tb.IdTipoBeneficio_NK,
        LEFT(tb.NombreBeneficio, 40) AS NombreBeneficio,
        tb.FechaInicioValidez,
        tb.FechaFinValidez,
        tb.EsActual,
        COUNT(h.CantidadOfertas) AS OfertasAsociadas
    FROM Dim_TipoBeneficio tb
    LEFT JOIN Hecho_OfertaPlanBeneficio h
        ON h.TipoBeneficioSK = tb.TipoBeneficioSK
    WHERE tb.IdTipoBeneficio_NK = '640'
    GROUP BY
        tb.TipoBeneficioSK,
        tb.IdTipoBeneficio_NK,
        tb.NombreBeneficio,
        tb.FechaInicioValidez,
        tb.FechaFinValidez,
        tb.EsActual
    ORDER BY tb.TipoBeneficioSK
    """
)


=== 1. Auditoría de conteos en Proyecto_202615_G6 ===
Tabla                     | Registros_MySQL | Esperado
------------------------------------------------------
Dim_Tiempo                | 3               | 3       
Dim_Proveedor             | 126             | 126     
Dim_Plan                  | 1168            | 1168    
Dim_CondicionPago         | 24              | 24      
Dim_AreaServicio          | 5410            | 5410    
Dim_TipoBeneficio         | 218             | 218     
Hecho_OfertaPlanBeneficio | 22963           | 22963   

=== 2. Verificación de métricas y comodines en Hecho_OfertaPlanBeneficio ===
TotalFilas | TotalOfertas | Fecha_Comodin | Area_Comodin | Limite_333_Imputado | Limite_NULL_Ilimitado | Limite_Explicito
-------------------------------------------------------------------------------------------------------------------------
22963      | 22963        | 1584          | 5000         | 5388                | 14092                 | 3483            

=== 3

### 2. Validación Analítica de los 4 Temas RaSA

In [13]:
consultar_mysql(
    "RaSA 1. Proveedores con variación > 20% frente al promedio de mercado (Top 15)",
    """
    SELECT
        t.Anio,
        p.IdProveedor_NK,
        LEFT(tb.NombreBeneficio, 35) AS Beneficio,
        ROUND(AVG(h.ValorCopago), 2) AS Copago_Prov,
        ROUND(m.PromCopago, 2) AS Copago_Mercado,
        ROUND(AVG(h.ValorCoseguro), 2) AS Coseguro_Prov,
        ROUND(m.PromCoseguro, 2) AS Coseguro_Mercado
    FROM Hecho_OfertaPlanBeneficio h
    JOIN Dim_Tiempo t ON h.FechaSK = t.FechaSK
    JOIN Dim_Proveedor p ON h.ProveedorSK = p.ProveedorSK
    JOIN Dim_TipoBeneficio tb ON h.TipoBeneficioSK = tb.TipoBeneficioSK
    JOIN (
        SELECT
            h2.FechaSK,
            tb2.IdTipoBeneficio_NK,
            AVG(h2.ValorCopago) AS PromCopago,
            AVG(h2.ValorCoseguro) AS PromCoseguro
        FROM Hecho_OfertaPlanBeneficio h2
        JOIN Dim_TipoBeneficio tb2 ON h2.TipoBeneficioSK = tb2.TipoBeneficioSK
        WHERE h2.FechaSK <> 0
        GROUP BY h2.FechaSK, tb2.IdTipoBeneficio_NK
    ) m ON h.FechaSK = m.FechaSK AND tb.IdTipoBeneficio_NK = m.IdTipoBeneficio_NK
    WHERE h.FechaSK <> 0 AND p.ProveedorSK <> 0
    GROUP BY t.Anio, p.IdProveedor_NK, tb.NombreBeneficio, m.PromCopago, m.PromCoseguro
    HAVING
        (m.PromCopago > 0 AND ABS(AVG(h.ValorCopago) - m.PromCopago) / m.PromCopago > 0.20)
        OR
        (m.PromCoseguro > 0 AND ABS(AVG(h.ValorCoseguro) - m.PromCoseguro) / m.PromCoseguro > 0.20)
    ORDER BY t.Anio, Copago_Prov DESC
    LIMIT 15
    """
)

consultar_mysql(
    "RaSA 2. Áreas de servicio con menor cobertura (< 2 proveedores distintos)",
    """
    SELECT
        t.Anio,
        a.Estado,
        a.IdAreaDeServicio_NK,
        LEFT(a.NombreAreaDeServicio, 35) AS AreaServicio,
        COUNT(DISTINCT h.ProveedorSK) AS ProveedoresDistintos,
        COUNT(DISTINCT h.PlanSK) AS PlanesDistintos,
        SUM(h.CantidadOfertas) AS TotalOfertas
    FROM Hecho_OfertaPlanBeneficio h
    JOIN Dim_Tiempo t ON h.FechaSK = t.FechaSK
    JOIN Dim_AreaServicio a ON h.AreaServicioSK = a.AreaServicioSK
    WHERE h.FechaSK <> 0 AND h.AreaServicioSK <> 0 AND h.ProveedorSK <> 0
    GROUP BY t.Anio, a.Estado, a.IdAreaDeServicio_NK, a.NombreAreaDeServicio
    HAVING COUNT(DISTINCT h.ProveedorSK) < 2
    ORDER BY t.Anio, TotalOfertas DESC
    LIMIT 15
    """
)

consultar_mysql(
    "RaSA 3. Concentración de planes por población (Planes por cada 100 mil hab.)",
    """
    SELECT
        t.Anio,
        a.Estado,
        a.CondadoPrincipal,
        LEFT(a.NombreAreaDeServicio, 30) AS AreaServicio,
        a.PoblacionActual,
        COUNT(DISTINCT h.PlanSK) AS PlanesOfertados,
        ROUND(COUNT(DISTINCT h.PlanSK) * 100000.0 / a.PoblacionActual, 4) AS PlanesPor100MilHab
    FROM Hecho_OfertaPlanBeneficio h
    JOIN Dim_Tiempo t ON h.FechaSK = t.FechaSK
    JOIN Dim_AreaServicio a ON h.AreaServicioSK = a.AreaServicioSK
    WHERE h.FechaSK <> 0 AND h.AreaServicioSK <> 0 AND a.PoblacionActual > 0
    GROUP BY t.Anio, a.Estado, a.CondadoPrincipal, a.NombreAreaDeServicio, a.PoblacionActual
    ORDER BY PlanesPor100MilHab DESC
    LIMIT 15
    """
)

consultar_mysql(
    "RaSA 4. Evolución anual de ofertas, copago y coseguro promedio (incluyendo comodín y SCD2)",
    """
    SELECT
        t.FechaSK,
        t.Anio,
        t.Fecha,
        SUM(h.CantidadOfertas) AS TotalOfertas,
        COUNT(DISTINCT h.ProveedorSK) AS ProveedoresActivos,
        COUNT(DISTINCT h.PlanSK) AS PlanesActivos,
        COUNT(DISTINCT h.TipoBeneficioSK) AS VersionesBeneficioUsadas,
        ROUND(AVG(h.ValorCopago), 2) AS CopagoPromedio,
        ROUND(AVG(h.ValorCoseguro), 2) AS CoseguroPromedio
    FROM Hecho_OfertaPlanBeneficio h
    JOIN Dim_Tiempo t ON h.FechaSK = t.FechaSK
    GROUP BY t.FechaSK, t.Anio, t.Fecha
    ORDER BY t.FechaSK
    """
)


=== RaSA 1. Proveedores con variación > 20% frente al promedio de mercado (Top 15) ===
Anio | IdProveedor_NK | Beneficio                           | Copago_Prov | Copago_Mercado | Coseguro_Prov | Coseguro_Mercado
-----------------------------------------------------------------------------------------------------------------------------
2017 | 26250          | Orthodontia - Adult                 | 1400.0      | 15.73          | 0.0           | 0.0             
2017 | 36096          | Outpatient Facility Fee (e.g.,  Amb | 825.0       | 61.54          | 32.5          | 31.19           
2017 | 36096          | Delivery and All Inpatient Services | 780.0       | 112.77         | 30.0          | 29.57           
2017 | 36096          | Substance Abuse Disorder Inpatient  | 700.0       | 109.93         | 26.0          | 31.35           
2017 | 36096          | Inpatient Hospital Services (e.g.,  | 680.0       | 81.91          | 24.0          | 29.22           
2017 | 36239          | Mental

### 3. Conclusiones de la Ejecución ETL

- El flujo ETL en PySpark cargó sin errores de integridad referencial las 6 dimensiones (`Dim_Tiempo`: 3, `Dim_Proveedor`: 126, `Dim_Plan`: 1,168, `Dim_CondicionPago`: 24, `Dim_AreaServicio`: 5,410 y `Dim_TipoBeneficio`: 218) y la tabla de hechos (`Hecho_OfertaPlanBeneficio`: 22,963 registros) en `Proyecto_202615_G6`.
- El manejo de comodines (`SK = 0`), la imputación condicional de `CantidadLimite = 333` (5,388 registros) y el historial **SCD Tipo 2** en `Dim_TipoBeneficio` (`ID = 640` dividido entre `SK = 136` con 195 ofertas en 2017 y `SK = 137` con 302 ofertas en 2018) quedaron verificados directamente en MySQL.
- Las cuatro consultas analíticas de RaSA confirman que el Data Mart soporta análisis de competitividad de proveedores, brechas de cobertura por área, concentración por población y evolución temporal entre 2017 y 2018.